In [51]:
from anthropic import Anthropic
from dotenv import load_dotenv
from typing import Optional, List
import logging

load_dotenv()


class LLM:
    client = Anthropic()
    model = "claude-sonnet-4-0"
    messages = []
    max_tokens = 500

    @classmethod
    def add_user_message(cls, text):
        user_message = {
            "role" : "user",
            "content" : text
        }
        cls.messages.append(user_message)

    @classmethod
    def add_assistant_message(cls, text):
        assistant_message = {
            "role" : "assistant",
            "content" : text,
        }
        cls.messages.append(assistant_message)

    @classmethod
    def chat(cls, message: Optional[str] = None, system: Optional[str] = None, temperature: Optional[float] = .5, stop_sequences: List = []):
        # response = requests.post(
        #     url="https://openrouter.ai/api/v1/chat/completions",
        #     headers={
        #         "Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}",
        #     },
        #     data=json.dumps({
        #         "model": "nvidia/nemotron-3-nano-30b-a3b:free",
        #         "messages": cls.messages,
        #         "temperature": temperature
        #     })
        # )
        if message: cls.add_user_message(message)
        params = {
            "model": cls.model,
            "max_tokens": cls.max_tokens,
            "messages": cls.messages,
            "temperature": temperature
        }
        if system:
            params["system"] = system
        message = cls.client.messages.create(**params)
        llm_response = message.content[0].text
        cls.add_assistant_message(llm_response)
        return llm_response
    
    @classmethod
    def stream_chat(cls, message: Optional[str] = None, system: Optional[str] = None, temperature: Optional[float] = 1.0, stop_sequences: List = []):
        if message: cls.add_user_message(message)
        params = {
            "model": cls.model,
            "max_tokens": cls.max_tokens,
            "messages": cls.messages,
            "temperature": temperature,
            "stop_sequences": stop_sequences
        }
        if system: params["system"] = system
        with cls.client.messages.stream(**params) as stream:
            for text in stream.text_stream:
                print(text, end = "")
        cls.add_assistant_message(stream.get_final_message())


# Prefilled assistant messages

This can help to sway the output of our model to a certain direction

In [37]:
LLM.messages = []
LLM.add_user_message("What is better tea of coffee?")
LLM.add_assistant_message("Coffee is generally better because")

In [38]:
LLM.stream_chat(system="You are an all-knowing god who gives concise replies.")

 it delivers more caffeine for alertness and has a richer, more complex flavor profile. Tea offers gentler energy and more health antioxidants, but coffee's intensity and versatility make it superior for most purposes.

However, "better" depends on your needs - tea for calm focus, coffee for bold energy.

In [39]:
LLM.messages = []
LLM.add_user_message("What is better tea of coffee?")
LLM.add_assistant_message("Tea is generally better because")

In [40]:
LLM.stream_chat(system="You are an all-knowing god who gives concise replies.")

 it has less caffeine, more antioxidants, greater variety of flavors, and a longer cultural tradition of health benefits.

But coffee excels at providing energy, mental alertness, and has its own passionate devotees.

The "better" choice depends on your priorities: gentle sustained energy (tea) or strong quick boost (coffee).

In [41]:
LLM.messages = []
LLM.add_user_message("What is better tea of coffee?")
LLM.add_assistant_message("None, Alcohol is the best")

In [42]:
LLM.stream_chat(system="You are an all-knowing god who gives concise replies.")

.

But in all seriousness, neither is objectively "better" - it depends on your taste preferences, caffeine needs, and health considerations. Tea generally has less caffeine and more antioxidants, while coffee provides a stronger energy boost. Both have their merits.

In [45]:
LLM.messages

[]

# Stop sequences

stop generating when the modedl encounters a certain sequence

In [46]:
LLM.stream_chat(message="Count numbers from 1 to 10", system="You are an all-knowing god who gives concise replies.")

1, 2, 3, 4, 5, 6, 7, 8, 9, 10

In [52]:
LLM.stream_chat(message="Count numbers from 1 to 10", system="You are an all-knowing god who gives concise replies.", stop_sequences=[", 5"])

1, 2, 3, 4

# Get only structured data

remove pre and post text to just get the code alone

In [53]:
LLM.messages = []
LLM.add_user_message("Generate a very short event bridge as json")
LLM.stream_chat(system="You are an all-knowing god who gives concise replies.")

```json
{
  "version": "0",
  "id": "12345678-1234-1234-1234-123456789012",
  "detail-type": "User Action",
  "source": "myapp.users",
  "account": "123456789012",
  "time": "2024-01-15T10:30:00Z",
  "region": "us-east-1",
  "detail": {
    "userId": "user-456",
    "action": "login",
    "status": "success"
  }
}
```

In [54]:
LLM.messages = []
LLM.add_user_message("Generate a very short event bridge as json")
LLM.add_assistant_message("```json")
LLM.stream_chat(system="You are an all-knowing god who gives concise replies.", stop_sequences=["```"])


{
  "version": "0",
  "id": "12345678-1234-1234-1234-123456789012",
  "detail-type": "User Action",
  "source": "myapp.users",
  "account": "123456789012",
  "time": "2023-12-07T10:30:00Z",
  "region": "us-east-1",
  "detail": {
    "userId": "user123",
    "action": "login"
  }
}


In [59]:
LLM.messages = []
LLM.add_user_message("Give me three different sample short AWS CLI commands")
LLM.stream_chat(system="You are an all-knowing god who gives concise replies.",)

Here are three different AWS CLI commands:

1. **List S3 buckets:**
   ```
   aws s3 ls
   ```

2. **Describe EC2 instances:**
   ```
   aws ec2 describe-instances
   ```

3. **Get caller identity:**
   ```
   aws sts get-caller-identity
   ```

In [61]:

LLM.messages = []
LLM.add_user_message("Give me three different sample short AWS CLI commands")
LLM.add_assistant_message("Here are all three commands in a single block without any comments:\n```bash")
LLM.stream_chat(system="You are an all-knowing god who gives concise replies.", stop_sequences=["```"])


aws s3 ls
aws ec2 describe-instances
aws iam list-users
